# Introduccion al Procesamiento de Lenguaje Natural (NLP)

**Duracion estimada:** ~2 horas  
**Nivel:** Introductorio  
**Requisitos previos:** Python basico, nociones de programacion

**Dataset:** [Amazon Reviews Multi ES](https://huggingface.co/datasets/SetFit/amazon_reviews_multi_es) - 200K resenas reales de Amazon en espanol con ratings de 1 a 5 estrellas

---

## Agenda

| Bloque | Tema | Duracion |
|--------|------|----------|
| 1 | Que es NLP y por que importa | 15 min |
| 2 | Exploracion del dataset | 15 min |
| 3 | Preprocesamiento de texto | 25 min |
| 4 | Representacion de texto: Bag of Words y TF-IDF | 20 min |
| 5 | Clasificacion de sentimiento con Machine Learning | 25 min |
| 6 | Introduccion a Word Embeddings | 15 min |
| 7 | Cierre y recursos adicionales | 5 min |

---
## 0. Instalacion de dependencias

In [ ]:
!pip install nltk scikit-learn matplotlib numpy pandas gensim datasets

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

print("Librerias cargadas correctamente.")

---
## 1. Que es NLP y por que importa (~15 min)

### Definicion

El **Procesamiento de Lenguaje Natural (NLP)** es un campo de la inteligencia artificial que se enfoca en la interaccion entre computadoras y el lenguaje humano. Su objetivo es que las maquinas puedan **entender, interpretar y generar** texto de manera util.

### Aplicaciones en el mundo real

- **Chatbots y asistentes virtuales** (Siri, Alexa, ChatGPT)
- **Traduccion automatica** (Google Translate, DeepL)
- **Analisis de sentimiento** (opiniones en redes sociales, resenas) <-- Lo que haremos hoy!
- **Resumen automatico** de textos
- **Extraccion de informacion** de documentos
- **Clasificacion de correos** (spam vs. no spam)
- **Busqueda semantica** (Google, motores de busqueda)

### Por que es dificil?

El lenguaje humano es:
- **Ambiguo**: "El banco esta cerca" (banco financiero o banco del parque?)
- **Contextual**: "Hace frio" puede ser una queja o una invitacion a cerrar la ventana
- **Variable**: jerga, regionalismos, errores ortograficos
- **Implicito**: mucho del significado no esta en las palabras explicitas

### Pipeline tipico de NLP

```
Texto crudo -> Preprocesamiento -> Representacion numerica -> Modelo -> Resultado
```

### Ejercicio de discusion

Piensen en 2-3 aplicaciones de NLP que usen en su dia a dia. Compartan con el grupo.

---
## 2. Exploracion del dataset (~15 min)

Vamos a trabajar con **resenas reales de Amazon en espanol**. El dataset contiene 200,000 resenas de productos con ratings de 1 a 5 estrellas (labels 0-4).

### 2.1 Cargar el dataset

In [ ]:
from datasets import load_dataset

# Cargar resenas de Amazon en espanol
dataset = load_dataset("SetFit/amazon_reviews_multi_es")
print(dataset)

In [ ]:
# Convertir a DataFrame y tomar una muestra manejable
df_full = dataset["train"].to_pandas()
df = df_full.sample(n=5000, random_state=42).reset_index(drop=True)

# El label va de 0 a 4, lo convertimos a estrellas 1-5 para que sea mas intuitivo
df["estrellas"] = df["label"] + 1

print(f"Muestra de trabajo: {len(df)} resenas")
print(f"Columnas: {list(df.columns)}")
df.head()

### 2.2 Explorar la estructura

In [ ]:
print("=== Informacion general ===")
print(f"Filas: {len(df)}")
print(f"\n=== Valores nulos ===")
print(df.isnull().sum())
print(f"\n=== Longitud promedio de resenas (caracteres) ===")
print(df["text"].str.len().describe().round(1))

In [ ]:
# Distribucion de ratings
print("=== Distribucion de estrellas ===")
print(df["estrellas"].value_counts().sort_index())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de ratings
colores_rating = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
conteos = df["estrellas"].value_counts().sort_index()
axes[0].bar(conteos.index, conteos.values, color=colores_rating, edgecolor='white')
axes[0].set_xlabel('Rating (estrellas)')
axes[0].set_ylabel('Cantidad de resenas')
axes[0].set_title('Distribucion de ratings')
axes[0].set_xticks([1, 2, 3, 4, 5])

# Distribucion de longitud de resenas
axes[1].hist(df["text"].str.len(), bins=50, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Longitud (caracteres)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribucion de longitud de resenas')

plt.tight_layout()
plt.show()

In [ ]:
# Ver algunas resenas de ejemplo
print("=== Ejemplos de resenas por rating ===")
for rating in [1, 3, 5]:
    print(f"\n--- {rating} estrella(s) ---")
    ejemplos_rating = df[df["estrellas"] == rating]["text"].head(3)
    for texto in ejemplos_rating:
        print(f"  {str(texto)[:120]}...")

### 2.3 Simplificar a sentimiento binario

Para esta sesion introductoria, vamos a simplificar los 5 ratings a **sentimiento binario**:
- Ratings 1-2 estrellas -> **negativo**
- Ratings 4-5 estrellas -> **positivo**
- Rating 3 -> lo descartamos (ambiguo)

In [ ]:
# Crear etiquetas de sentimiento
df_sentimiento = df[df["estrellas"] != 3].copy()
df_sentimiento["sentimiento"] = df_sentimiento["estrellas"].apply(
    lambda x: "positivo" if x >= 4 else "negativo"
)

# Eliminar resenas vacias
df_sentimiento = df_sentimiento.dropna(subset=["text"])
df_sentimiento = df_sentimiento[df_sentimiento["text"].str.strip() != ""].reset_index(drop=True)

print(f"Resenas con sentimiento: {len(df_sentimiento)}")
print(f"\nDistribucion:")
print(df_sentimiento["sentimiento"].value_counts())

---
## 3. Preprocesamiento de texto (~25 min)

Antes de que un modelo pueda trabajar con texto, necesitamos **limpiarlo y normalizarlo**. Este es uno de los pasos mas importantes en cualquier proyecto de NLP.

### 3.1 Limpieza basica

Pasos comunes:
1. Convertir a minusculas
2. Eliminar caracteres especiales y numeros
3. Eliminar espacios extra

In [ ]:
def limpiar_texto(texto):
    """Limpieza basica de texto."""
    texto = str(texto)
    # Convertir a minusculas
    texto = texto.lower()
    # Eliminar caracteres especiales (mantener letras, espacios y caracteres espanoles)
    texto = re.sub(r'[^a-z\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f1\s]', '', texto)
    # Eliminar espacios multiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# Aplicar a unas resenas de ejemplo
ejemplos = df_sentimiento["text"].head(5).tolist()

print("=== Antes y despues de limpiar ===")
for original in ejemplos:
    limpio = limpiar_texto(original)
    print(f"ORIGINAL: {str(original)[:100]}")
    print(f"LIMPIO:   {limpio[:100]}")
    print()

### 3.2 Tokenizacion

**Tokenizar** es dividir el texto en unidades individuales (tokens). Generalmente, cada token es una palabra.

In [ ]:
from nltk.tokenize import word_tokenize

ejemplo = limpiar_texto(ejemplos[0])
tokens = word_tokenize(ejemplo, language='spanish')

print(f"Texto: {ejemplo}")
print(f"Tokens: {tokens}")
print(f"Numero de tokens: {len(tokens)}")

### 3.3 Stopwords

Las **stopwords** son palabras muy comunes que generalmente no aportan significado ("el", "la", "de", "que", etc.).

In [ ]:
from nltk.corpus import stopwords

stop_words_es = set(stopwords.words('spanish'))

print(f"Numero de stopwords en espanol: {len(stop_words_es)}")
print(f"Ejemplos: {sorted(list(stop_words_es))[:20]}")

In [ ]:
def eliminar_stopwords(tokens, stop_words):
    """Elimina stopwords de una lista de tokens."""
    return [t for t in tokens if t not in stop_words]

tokens_ejemplo = word_tokenize(limpiar_texto(ejemplos[0]), language='spanish')
tokens_filtrados = eliminar_stopwords(tokens_ejemplo, stop_words_es)

print(f"Tokens originales:  {tokens_ejemplo}")
print(f"Sin stopwords:      {tokens_filtrados}")

### 3.4 Stemming

**Stemming** reduce las palabras a su raiz cortando sufijos. Es rapido pero a veces agresivo.

Ejemplo: "comprando", "comprado", "compras" -> "compr"

In [ ]:
from nltk.stem import SnowballStemmer

stemmer = SnowballStemmer('spanish')

palabras_ejemplo = ['comprado', 'comprando', 'compras', 'compre',
                     'funcionando', 'funciona', 'recomiendo', 'recomendado']

print(f"{'Palabra':<15} {'Stem':<15}")
print("-" * 30)
for palabra in palabras_ejemplo:
    print(f"{palabra:<15} {stemmer.stem(palabra):<15}")

### 3.5 Pipeline completo de preprocesamiento

In [ ]:
def preprocesar(texto):
    """Pipeline completo de preprocesamiento."""
    texto = limpiar_texto(texto)
    tokens = word_tokenize(texto, language='spanish')
    tokens = eliminar_stopwords(tokens, stop_words_es)
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

print("=== Preprocesamiento de resenas reales de Amazon ===")
for original in ejemplos[:4]:
    procesado = preprocesar(original)
    print(f"ORIGINAL:   {str(original)[:100]}")
    print(f"PROCESADO:  {procesado}")
    print()

### 3.6 Analisis de frecuencia de palabras

In [ ]:
# Analizar las palabras mas frecuentes por sentimiento
palabras_pos = Counter()
palabras_neg = Counter()

muestra = df_sentimiento.sample(n=min(2000, len(df_sentimiento)), random_state=42)
for _, row in muestra.iterrows():
    tokens = preprocesar(row["text"])
    if row["sentimiento"] == "positivo":
        palabras_pos.update(tokens)
    else:
        palabras_neg.update(tokens)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top palabras positivas
top_pos = palabras_pos.most_common(15)
palabras_p, freqs_p = zip(*top_pos)
axes[0].barh(range(len(palabras_p)), freqs_p, color='#2ecc71')
axes[0].set_yticks(range(len(palabras_p)))
axes[0].set_yticklabels(palabras_p)
axes[0].set_title('Top 15 palabras - Resenas POSITIVAS')
axes[0].invert_yaxis()

# Top palabras negativas
top_neg = palabras_neg.most_common(15)
palabras_n, freqs_n = zip(*top_neg)
axes[1].barh(range(len(palabras_n)), freqs_n, color='#e74c3c')
axes[1].set_yticks(range(len(palabras_n)))
axes[1].set_yticklabels(palabras_n)
axes[1].set_title('Top 15 palabras - Resenas NEGATIVAS')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### Ejercicio practico 1

Modifica la funcion `preprocesar` para que **no** use stemming. Luego compara las palabras mas frecuentes con y sin stemming. Que diferencias notas?

In [ ]:
# Tu codigo aqui
def preprocesar_sin_stem(texto):
    """Pipeline sin stemming."""
    pass  # Implementa aqui

# Prueba con una resena
# print(preprocesar_sin_stem(ejemplos[0]))

---
## 4. Representacion de texto: BoW y TF-IDF (~20 min)

Los modelos de machine learning necesitan **numeros**, no texto. Necesitamos convertir texto en vectores numericos.

### 4.1 Bag of Words (BoW)

La idea mas simple: contar cuantas veces aparece cada palabra en un documento.

```
"el gato come pescado" -> {el: 1, gato: 1, come: 1, pescado: 1}
"el perro come carne"  -> {el: 1, perro: 1, come: 1, carne: 1}
```

**Limitacion**: ignora el orden de las palabras y no captura significado.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Preparar textos limpios
textos_limpios = [limpiar_texto(t) for t in df_sentimiento["text"].tolist()]

# Crear el vectorizador BoW
bow_vectorizer = CountVectorizer(max_features=1000)
bow_matrix = bow_vectorizer.fit_transform(textos_limpios)

vocabulario = bow_vectorizer.get_feature_names_out()
print(f"Tamano del vocabulario: {len(vocabulario)}")
print(f"Primeras 20 palabras: {vocabulario[:20].tolist()}")
print(f"\nForma de la matriz: {bow_matrix.shape} (documentos x palabras)")

In [ ]:
# Visualizar la representacion de los primeros documentos
df_bow = pd.DataFrame(
    bow_matrix[:5].toarray(),
    columns=vocabulario,
    index=[f"doc_{i}" for i in range(5)]
)

# Mostrar solo columnas con valores > 0
cols_no_cero = df_bow.columns[(df_bow > 0).any()]
print(f"Palabras activas en los primeros 5 docs: {len(cols_no_cero)}")
df_bow[cols_no_cero]

### 4.2 TF-IDF (Term Frequency - Inverse Document Frequency)

Mejora sobre BoW: no solo cuenta frecuencia, sino que **penaliza palabras que aparecen en muchos documentos** (son menos informativas).

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

- **TF(t, d)**: Frecuencia del termino `t` en el documento `d`
- **IDF(t)**: $\log\left(\frac{N}{\text{num docs que contienen } t}\right)$ — penaliza palabras muy comunes

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf_vectorizer.fit_transform(textos_limpios)

print(f"Forma de la matriz TF-IDF: {tfidf_matrix.shape}")

# Top palabras por TF-IDF para algunos documentos
tfidf_vocab = tfidf_vectorizer.get_feature_names_out()

print("\n=== Top 5 palabras por TF-IDF en las primeras 5 resenas ===")
for i in range(5):
    scores = tfidf_matrix[i].toarray().flatten()
    top_indices = scores.argsort()[-5:][::-1]
    top_palabras = [(tfidf_vocab[j], scores[j]) for j in top_indices]
    sent = df_sentimiento.iloc[i]["sentimiento"]
    top_str = ", ".join([f"{p}({s:.3f})" for p, s in top_palabras])
    print(f"Doc {i} [{sent:>8}]: {top_str}")

### Ejercicio practico 2

Crea un `TfidfVectorizer` que:
1. Ignore palabras que aparecen en mas del 80% de los documentos (`max_df=0.8`)
2. Ignore palabras que aparecen en menos de 5 documentos (`min_df=5`)
3. Considere unigramas y bigramas (`ngram_range=(1, 2)`)

Observa como cambia el vocabulario.

In [ ]:
# Tu codigo aqui
# tfidf_v2 = TfidfVectorizer(max_df=???, min_df=???, ngram_range=???)
# ...

---
## 5. Clasificacion de sentimiento con Machine Learning (~25 min)

Ahora vamos a construir un clasificador de sentimiento **con datos reales de Amazon**.

### 5.1 Preparar los datos

In [ ]:
from sklearn.model_selection import train_test_split

# Vectorizar con TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, max_df=0.9, min_df=2)
X = vectorizer.fit_transform(textos_limpios)
y = np.array([1 if s == "positivo" else 0 for s in df_sentimiento["sentimiento"]])

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Entrenamiento: {X_train.shape[0]} resenas")
print(f"Prueba: {X_test.shape[0]} resenas")
print(f"Dimensiones del vector: {X_train.shape[1]} features")
print(f"\nBalance de clases (train):")
print(f"  Positivo: {(y_train == 1).sum()} ({(y_train == 1).mean():.1%})")
print(f"  Negativo: {(y_train == 0).sum()} ({(y_train == 0).mean():.1%})")

### 5.2 Entrenar y comparar modelos

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

modelos = {
    "Naive Bayes": MultinomialNB(),
    "Regresion Logistica": LogisticRegression(max_iter=1000, random_state=42),
    "SVM Lineal": LinearSVC(max_iter=1000, random_state=42),
}

resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    resultados[nombre] = acc
    print(f"\n{'='*50}")
    print(f"Modelo: {nombre}")
    print(f"Accuracy: {acc:.2%}")
    print(classification_report(y_test, y_pred, target_names=["negativo", "positivo"]))

In [ ]:
# Comparar modelos visualmente
plt.figure(figsize=(8, 4))
nombres = list(resultados.keys())
accuracies = list(resultados.values())
colores = ['#2ecc71', '#3498db', '#e74c3c']

bars = plt.bar(nombres, accuracies, color=colores, edgecolor='white', linewidth=1.5)
plt.ylabel('Accuracy')
plt.title('Comparacion de modelos - Sentimiento en resenas de Amazon (espanol)')
plt.ylim(0, 1.1)

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{acc:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 5.3 Probar con nuestras propias resenas

In [ ]:
mejor_modelo = modelos["Regresion Logistica"]

nuevas_resenas = [
    "Excelente producto, me encanto la calidad!",
    "No sirve para nada, muy mala calidad",
    "Llego rapido pero el producto no es como en las fotos",
    "Muy bueno por el precio, lo recomiendo",
    "Pesimo, pedi la devolucion inmediatamente",
    "Funciona bien, cumple con lo esperado",
]

X_nuevas = vectorizer.transform([limpiar_texto(r) for r in nuevas_resenas])
predicciones = mejor_modelo.predict(X_nuevas)

print("=== Predicciones para nuevas resenas ===")
for resena, pred in zip(nuevas_resenas, predicciones):
    sentimiento = "POSITIVO" if pred == 1 else "NEGATIVO"
    print(f"[{sentimiento:>8}] {resena}")

### 5.4 Que palabras son mas importantes para el modelo?

In [ ]:
# Coeficientes de la Regresion Logistica
feature_names = vectorizer.get_feature_names_out()
coeficientes = mejor_modelo.coef_[0]

top_k = 15
top_pos_idx = coeficientes.argsort()[-top_k:][::-1]
top_neg_idx = coeficientes.argsort()[:top_k]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Palabras mas positivas
palabras_pos_model = [feature_names[i] for i in top_pos_idx]
coefs_pos = [coeficientes[i] for i in top_pos_idx]
axes[0].barh(range(top_k), coefs_pos, color='#2ecc71')
axes[0].set_yticks(range(top_k))
axes[0].set_yticklabels(palabras_pos_model)
axes[0].set_title('Palabras mas asociadas a POSITIVO')
axes[0].invert_yaxis()

# Palabras mas negativas
palabras_neg_model = [feature_names[i] for i in top_neg_idx]
coefs_neg = [coeficientes[i] for i in top_neg_idx]
axes[1].barh(range(top_k), coefs_neg, color='#e74c3c')
axes[1].set_yticks(range(top_k))
axes[1].set_yticklabels(palabras_neg_model)
axes[1].set_title('Palabras mas asociadas a NEGATIVO')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### Ejercicio practico 3

1. Escribe 3-5 resenas tuyas (como si compraras en Amazon) y prueba el clasificador
2. Intenta encontrar un caso donde falle (sarcasmo, negacion, etc.)
3. Por que crees que falla en esos casos?

In [ ]:
# Tu codigo aqui
mis_resenas = [
    # Escribe tus resenas aqui
]

# X_mis = vectorizer.transform([limpiar_texto(r) for r in mis_resenas])
# predicciones_mis = mejor_modelo.predict(X_mis)
# for resena, pred in zip(mis_resenas, predicciones_mis):
#     sentimiento = "POSITIVO" if pred == 1 else "NEGATIVO"
#     print(f"[{sentimiento:>8}] {resena}")

### Ejercicio practico 4 (bonus)

En vez de sentimiento binario, intenta clasificar las **5 estrellas** directamente (multiclase). Como cambia el rendimiento? Cuales son las clases mas dificiles de distinguir?

In [ ]:
# Tu codigo aqui
# Pista: usa df en vez de df_sentimiento, y label como target
# ...

---
## 6. Introduccion a Word Embeddings (~15 min)

### Limitaciones de BoW y TF-IDF

- No capturan **relaciones semanticas** entre palabras
- "excelente" y "genial" son igual de diferentes que "excelente" y "horrible"
- Vectores **dispersos** (sparse) y de alta dimension

### Que son los Word Embeddings?

Son representaciones **densas** de palabras en un espacio vectorial de baja dimension (100-300 dimensiones) donde **palabras con significado similar estan cerca** entre si.

```
rey - hombre + mujer = reina
paris - francia + espana = madrid
```

### Modelos populares:
- **Word2Vec** (Google, 2013)
- **GloVe** (Stanford, 2014)
- **FastText** (Facebook, 2016)

### 6.1 Entrenando Word2Vec con resenas de Amazon

In [ ]:
from gensim.models import Word2Vec

# Tokenizar todas las resenas
corpus_tokenizado = [
    word_tokenize(limpiar_texto(doc), language='spanish')
    for doc in df_sentimiento["text"].tolist()
]

print(f"Documentos en el corpus: {len(corpus_tokenizado)}")
print(f"Ejemplo: {corpus_tokenizado[0][:10]}...")

In [ ]:
# Entrenar Word2Vec
modelo_w2v = Word2Vec(
    sentences=corpus_tokenizado,
    vector_size=100,
    window=5,
    min_count=3,
    workers=4,
    epochs=30,
    seed=42,
)

print(f"Vocabulario del modelo: {len(modelo_w2v.wv)} palabras")
print(f"Dimensiones del vector: {modelo_w2v.wv.vector_size}")

In [ ]:
# Palabras similares (semantica aprendida de las resenas!)
print("=== Palabras mas similares (aprendido de resenas de Amazon) ===")
for palabra in ["excelente", "malo", "producto", "envio", "precio"]:
    if palabra in modelo_w2v.wv:
        similares = modelo_w2v.wv.most_similar(palabra, topn=5)
        sim_str = ", ".join([f"{p}({s:.2f})" for p, s in similares])
        print(f"'{palabra}' -> {sim_str}")
    else:
        print(f"'{palabra}' no esta en el vocabulario")
    print()

### 6.2 Visualizacion de embeddings con PCA

In [ ]:
from sklearn.decomposition import PCA
from matplotlib.patches import Patch

# Palabras interesantes para visualizar
palabras_interes = [
    "excelente", "bueno", "genial", "perfecto", "recomiendo",
    "malo", "horrible", "pesimo", "terrible", "basura",
    "producto", "calidad", "envio", "precio", "compra",
]

# Filtrar las que existan en el vocabulario
palabras_validas = [p for p in palabras_interes if p in modelo_w2v.wv]
vectores = np.array([modelo_w2v.wv[p] for p in palabras_validas])

# Reducir a 2D
pca = PCA(n_components=2)
vectores_2d = pca.fit_transform(vectores)

plt.figure(figsize=(12, 8))

positivas = {"excelente", "bueno", "genial", "perfecto", "recomiendo"}
negativas = {"malo", "horrible", "pesimo", "terrible", "basura"}

for i, palabra in enumerate(palabras_validas):
    x, y_coord = vectores_2d[i]
    if palabra in positivas:
        color = '#2ecc71'
    elif palabra in negativas:
        color = '#e74c3c'
    else:
        color = '#3498db'
    plt.scatter(x, y_coord, c=color, s=100, zorder=5)
    plt.annotate(palabra, (x, y_coord), fontsize=11, fontweight='bold',
                 xytext=(5, 5), textcoords='offset points')

plt.title('Visualizacion 2D de Word Embeddings (entrenados con resenas de Amazon)')
plt.xlabel('Componente 1')
plt.ylabel('Componente 2')
plt.grid(True, alpha=0.3)

leyenda = [
    Patch(facecolor='#2ecc71', label='Positivas'),
    Patch(facecolor='#e74c3c', label='Negativas'),
    Patch(facecolor='#3498db', label='Neutras/Tema'),
]
plt.legend(handles=leyenda, loc='best')
plt.tight_layout()
plt.show()

### 6.3 De palabras a documentos

Para representar un documento completo con embeddings, una tecnica simple es **promediar los vectores** de todas sus palabras.

In [ ]:
def documento_a_vector(tokens, modelo):
    """Convierte un documento a un vector promediando word embeddings."""
    vectores = [modelo.wv[t] for t in tokens if t in modelo.wv]
    if len(vectores) == 0:
        return np.zeros(modelo.wv.vector_size)
    return np.mean(vectores, axis=0)

# Ejemplo
tokens_ej = word_tokenize(
    limpiar_texto(df_sentimiento.iloc[0]["text"]), language='spanish'
)
vec_doc = documento_a_vector(tokens_ej, modelo_w2v)

print(f"Resena: {str(df_sentimiento.iloc[0]['text'])[:80]}...")
print(f"Vector (primeros 10): {vec_doc[:10].round(4)}")
print(f"Dimension: {len(vec_doc)}")

### Ejercicio practico 5

Usa los vectores de `documento_a_vector` como features para entrenar un clasificador. Compara con TF-IDF de la seccion 5. Cual funciona mejor? Por que?

In [ ]:
# Tu codigo aqui
# 1. Convertir todas las resenas a vectores
# X_w2v = np.array([documento_a_vector(tokens, modelo_w2v) for tokens in corpus_tokenizado])
# 2. Dividir en train/test
# 3. Entrenar un modelo (ej: LogisticRegression)
# 4. Evaluar y comparar

---
## 7. Cierre y recursos adicionales (~5 min)

### Resumen de lo aprendido

| Concepto | Descripcion |
|----------|-------------|
| **Dataset real** | 200K resenas de Amazon en espanol |
| **Preprocesamiento** | Limpieza, tokenizacion, stopwords, stemming |
| **BoW** | Representacion simple contando palabras |
| **TF-IDF** | BoW mejorado que penaliza palabras muy comunes |
| **Clasificacion** | Naive Bayes, Regresion Logistica, SVM para sentimiento |
| **Word Embeddings** | Representaciones densas que capturan semantica |

### Lo que NO cubrimos (temas para futuras sesiones)

- **Redes neuronales recurrentes (RNN, LSTM)** para secuencias
- **Transformers y atencion** (BERT, GPT, etc.)
- **NER** (Named Entity Recognition)
- **Generacion de texto** con modelos de lenguaje
- **Transfer learning** con modelos pre-entrenados (BETO, RoBERTa)

### Recursos para seguir aprendiendo

- **Libros:**
  - *Speech and Language Processing* - Jurafsky & Martin (gratis online)
  - *Natural Language Processing with Python* - Bird, Klein & Loper

- **Cursos:**
  - Stanford CS224N: NLP with Deep Learning
  - Hugging Face NLP Course (gratuito)

- **Librerias:**
  - `spaCy` - NLP industrial
  - `Hugging Face Transformers` - Modelos pre-entrenados
  - `NLTK` - Herramientas educativas

### Desafio para casa

Usando el dataset completo (200K resenas):

1. Entrena con **todas** las resenas (no solo la muestra de 5000)
2. Intenta la clasificacion **multiclase** (5 estrellas) y analiza la matriz de confusion
3. Experimenta con bigramas y trigramas en TF-IDF
4. Compara Word2Vec entrenado con 5K vs 200K resenas: mejoran los embeddings?

Comparte tus resultados en la proxima sesion!